# 07 地震 · 精细清洗（探查）

本阶段只读取并探查 `07_fine_clean` 内的原始数据，**不修改任何值，不导出清洗结果**。本地与 USGS 分开查看。


## STEP 1 · 路径

METHOD: 用 `pathlib.Path` 定位本文件夹。优先 `C:\Users\WHB\Desktop\DPW I\07_fine_clean`，若不存在则回退到当前工作目录或箱子上的同名目录。输入文件为 `ProjectData/07earthquake.csv` 与 `acquired/07_usgs_quakes_m5_2022_2023.csv`。


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

CANDIDATES = [
    Path(r"C:\Users\WHB\Desktop\DPW I\07_fine_clean"),
    Path.cwd(),
    Path("/workspace/07_fine_clean"),
]

ROOT = None
for cand in CANDIDATES:
    if (cand / "ProjectData" / "07earthquake.csv").exists():
        ROOT = cand
        break

if ROOT is None:
    raise FileNotFoundError("未找到 07_fine_clean/ProjectData/07earthquake.csv，请在该文件夹下运行本笔记本。")

LOCAL_IN = ROOT / "ProjectData" / "07earthquake.csv"
USGS_IN = ROOT / "acquired" / "07_usgs_quakes_m5_2022_2023.csv"
print("ROOT:", ROOT)
print("local exists:", LOCAL_IN.exists(), LOCAL_IN)
print("usgs  exists:", USGS_IN.exists(), USGS_IN)

## STEP 2 · 读取本地原始数据

METHOD: `pd.read_csv` 读取本地 CSV，不改列名，不删行。


In [ ]:
df_local = pd.read_csv(LOCAL_IN)
print("已读取:", LOCAL_IN.name)
df_local.head(3)

## STEP 3 · 本地：行数、列数、列名

METHOD: 打印 `shape` 与 `columns`，并查看前几行。


In [ ]:
print("local shape:", df_local.shape)
print("local columns:", list(df_local.columns))

## STEP 4 · 本地：各列类型与缺失

METHOD: `dtypes` 与 `isna().sum()`。缺失只计数，不填补、不删除。


In [ ]:
print("local dtypes:")
print(df_local.dtypes)
print()
print("local missing:")
print(df_local.isna().sum())

## STEP 5 · 本地 Date / Time 概览

METHOD: 查看 `Date`、`Time` 的字符串样本与长度分布，统计疑似 ISO（字符串长度 > 10）的行数。为检查可解析性，用临时 Series 做 `pd.to_datetime(..., format="mixed", utc=True, errors="coerce")`，**不写回**数据框。


In [ ]:
print("Date 样本:")
print(df_local["Date"].head(8).tolist())
print()
print("Time 样本:")
print(df_local["Time"].head(8).tolist())
print()
date_len = df_local["Date"].astype(str).str.len()
print("Date 字符串长度计数:")
print(date_len.value_counts().sort_index())
n_iso_like = int((date_len > 10).sum())
print("疑似 ISO 的 Date 行数 (len>10):", n_iso_like)

date_tmp = pd.to_datetime(df_local["Date"], format="mixed", utc=True, errors="coerce")
print("临时解析失败行数:", int(date_tmp.isna().sum()))
print("临时解析成功后的时间范围:", date_tmp.min(), "→", date_tmp.max())

## STEP 6 · 本地 Type 频数

METHOD: `value_counts(dropna=False)`，只观察，不筛选。


In [ ]:
print("local Type:")
print(df_local["Type"].value_counts(dropna=False))

## STEP 7 · 本地 Latitude / Longitude 范围与缺失

METHOD: 打印缺失数、最小值、最大值，并计数是否超出纬度 [-90, 90]、经度 [-180, 180]。不改值。


In [ ]:
for col, lo, hi in [("Latitude", -90, 90), ("Longitude", -180, 180)]:
    s = df_local[col]
    n_miss = int(s.isna().sum())
    n_out = int(((s < lo) | (s > hi)).sum())
    print(f"{col}: missing={n_miss}, min={s.min()}, max={s.max()}, out_of_range={n_out}")

## STEP 8 · 本地 Depth 范围、负值与缺失

METHOD: 统计缺失、最小值、最大值、`Depth < 0` 的行数。只计数，不置空、不删行。


In [ ]:
s = df_local["Depth"]
print("Depth missing:", int(s.isna().sum()))
print("Depth min / max:", s.min(), "/", s.max())
print("Depth < 0:", int((s < 0).sum()))
print("Depth > 700:", int((s > 700).sum()))

## STEP 9 · 本地 Magnitude 范围与缺失

METHOD: 统计缺失、最小值、最大值。只观察，不设阈值、不删行。


In [ ]:
s = df_local["Magnitude"]
print("Magnitude missing:", int(s.isna().sum()))
print("Magnitude min / max:", s.min(), "/", s.max())
print(s.describe())

## STEP 10 · 本地完全重复行

METHOD: `duplicated()` 统计完全相同的重复行。`keep="first"` 给出多余行数，`keep=False` 给出涉及重复的全部行数。不删除。


In [ ]:
n_extra = int(df_local.duplicated(keep="first").sum())
n_involved = int(df_local.duplicated(keep=False).sum())
print("完全重复多余行数 (keep=first):", n_extra)
print("涉及完全重复的全部行数 (keep=False):", n_involved)

## STEP 11 · 本地轻量图

METHOD: seaborn 画 Type 计数柱状图，以及 Magnitude 直方图。仅用于观察，不据此删除数据。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.countplot(data=df_local, y="Type", order=df_local["Type"].value_counts().index, ax=axes[0])
axes[0].set_title("local Type")
sns.histplot(data=df_local, x="Magnitude", bins=30, ax=axes[1])
axes[1].set_title("local Magnitude")
plt.tight_layout()
plt.show()

## STEP 12 · 读取 USGS 原始数据

METHOD: `pd.read_csv` 读取 USGS CSV，**暂不重命名列**，保持原始字段，便于对照探查。


In [ ]:
df_usgs = pd.read_csv(USGS_IN)
print("已读取:", USGS_IN.name)
df_usgs.head(3)

## STEP 13 · USGS：行数、列数、列名

METHOD: 打印 `shape` 与全部 `columns`。


In [ ]:
print("usgs shape:", df_usgs.shape)
print("usgs columns:", list(df_usgs.columns))

## STEP 14 · USGS：各列类型与缺失

METHOD: `dtypes` 与 `isna().sum()`。缺失只计数，不填补、不删除。


In [ ]:
print("usgs dtypes:")
print(df_usgs.dtypes)
print()
print("usgs missing:")
print(df_usgs.isna().sum())

## STEP 15 · USGS time 概览

METHOD: 查看 `time` 样本与字符串长度。同样用临时 Series 解析，不写回数据框。


In [ ]:
print("time 样本:")
print(df_usgs["time"].head(8).tolist())
time_len = df_usgs["time"].astype(str).str.len()
print("time 字符串长度计数:")
print(time_len.value_counts().sort_index())
time_tmp = pd.to_datetime(df_usgs["time"], format="mixed", utc=True, errors="coerce")
print("临时解析失败行数:", int(time_tmp.isna().sum()))
print("临时解析成功后的时间范围:", time_tmp.min(), "→", time_tmp.max())

## STEP 16 · USGS type 频数

METHOD: `value_counts(dropna=False)`，只观察，不筛选。


In [ ]:
print("usgs type:")
print(df_usgs["type"].value_counts(dropna=False))

## STEP 17 · USGS latitude / longitude 范围与缺失

METHOD: 与本地相同的范围检查。不改值。


In [ ]:
for col, lo, hi in [("latitude", -90, 90), ("longitude", -180, 180)]:
    s = df_usgs[col]
    n_miss = int(s.isna().sum())
    n_out = int(((s < lo) | (s > hi)).sum())
    print(f"{col}: missing={n_miss}, min={s.min()}, max={s.max()}, out_of_range={n_out}")

## STEP 18 · USGS depth 范围、负值与缺失

METHOD: 统计缺失、最小值、最大值、`depth < 0` 与 `depth > 700`。只计数，不置空、不删行。


In [ ]:
s = df_usgs["depth"]
print("depth missing:", int(s.isna().sum()))
print("depth min / max:", s.min(), "/", s.max())
print("depth < 0:", int((s < 0).sum()))
print("depth > 700:", int((s > 700).sum()))

## STEP 19 · USGS mag 范围与缺失

METHOD: 统计缺失、最小值、最大值。只观察。


In [ ]:
s = df_usgs["mag"]
print("mag missing:", int(s.isna().sum()))
print("mag min / max:", s.min(), "/", s.max())
print(s.describe())

## STEP 20 · USGS 完全重复行

METHOD: 与本地相同，用 `duplicated()` 计数，不删除。


In [ ]:
n_extra = int(df_usgs.duplicated(keep="first").sum())
n_involved = int(df_usgs.duplicated(keep=False).sum())
print("完全重复多余行数 (keep=first):", n_extra)
print("涉及完全重复的全部行数 (keep=False):", n_involved)

## STEP 21 · USGS 轻量图

METHOD: seaborn 画 type 计数柱状图，以及 mag 直方图。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.countplot(data=df_usgs, y="type", order=df_usgs["type"].value_counts().index, ax=axes[0])
axes[0].set_title("usgs type")
sns.histplot(data=df_usgs, x="mag", bins=30, ax=axes[1])
axes[1].set_title("usgs mag")
plt.tight_layout()
plt.show()

## STEP 22 · 探查小结

METHOD: 汇总两边的行数、完全重复、负深度、坐标越界、时间临时解析失败数。仍不导出、不改数据。下一步的清洗规则需再确认后才开始。


In [ ]:
print("—— 探查小结（未改数据、未导出）——")
print(f"local: {len(df_local)} 行 × {df_local.shape[1]} 列")
print(f"usgs : {len(df_usgs)} 行 × {df_usgs.shape[1]} 列")
print("local 完全重复多余行:", int(df_local.duplicated().sum()),
      "| Type:", df_local["Type"].value_counts().to_dict(),
      "| Depth<0:", int((df_local["Depth"] < 0).sum()),
      "| Mag:", df_local["Magnitude"].min(), "–", df_local["Magnitude"].max())
print("usgs  完全重复多余行:", int(df_usgs.duplicated().sum()),
      "| type:", df_usgs["type"].value_counts().to_dict(),
      "| depth<0:", int((df_usgs["depth"] < 0).sum()),
      "| mag:", df_usgs["mag"].min(), "–", df_usgs["mag"].max())

# 日期解析（已确认）

以下规则已确认后再写入：`pd.to_datetime(..., format="mixed", utc=True, errors="coerce")`，统一落到 UTC；解析失败记为缺失、不删行。本地只解析 `Date` 为 `Date_parsed`，不改写 `Time`；USGS 的 `time` 同样解析为 `Date_parsed`。只加列，不导出清洗结果。


## STEP 23 · 本地日期解析

METHOD: 对 `df_local["Date"]` 使用 `pd.to_datetime(..., format="mixed", utc=True, errors="coerce")`，结果写入新列 `Date_parsed`。不改 `Date`、不改 `Time`、不删行。


In [ ]:
n_local_before = len(df_local)
time_before = df_local["Time"].copy()
df_local["Date_parsed"] = pd.to_datetime(
    df_local["Date"], format="mixed", utc=True, errors="coerce"
)
print("已新增列 Date_parsed")
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)
print("Time 是否被改写:", not df_local["Time"].equals(time_before))

## STEP 24 · 检查本地 Date_parsed

METHOD: 统计解析失败数、时间范围，并列出原先 `Date` 字符串长度大于 10 的行（疑似 ISO），对照 `Date`、`Time`、`Date_parsed`。


In [ ]:
n_fail = int(df_local["Date_parsed"].isna().sum())
print("本地 Date_parsed 失败行数:", n_fail)
print("本地 Date_parsed 范围:", df_local["Date_parsed"].min(), "→", df_local["Date_parsed"].max())
print()
iso_mask = df_local["Date"].astype(str).str.len() > 10
print("疑似 ISO 行数:", int(iso_mask.sum()))
print(df_local.loc[iso_mask, ["Date", "Time", "Date_parsed"]])

## STEP 25 · USGS 日期解析

METHOD: 对 `df_usgs["time"]` 使用同一套 `pd.to_datetime(..., format="mixed", utc=True, errors="coerce")`，结果写入新列 `Date_parsed`。不改原始 `time`、不删行。


In [ ]:
n_usgs_before = len(df_usgs)
time_raw_before = df_usgs["time"].copy()
df_usgs["Date_parsed"] = pd.to_datetime(
    df_usgs["time"], format="mixed", utc=True, errors="coerce"
)
print("已新增列 Date_parsed")
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)
print("原始 time 是否被改写:", not df_usgs["time"].equals(time_raw_before))

## STEP 26 · 检查 USGS Date_parsed

METHOD: 统计解析失败数与时间范围，并抽查 `time` 与 `Date_parsed` 的对照样本。


In [ ]:
n_fail = int(df_usgs["Date_parsed"].isna().sum())
print("USGS Date_parsed 失败行数:", n_fail)
print("USGS Date_parsed 范围:", df_usgs["Date_parsed"].min(), "→", df_usgs["Date_parsed"].max())
print()
print(df_usgs[["time", "Date_parsed"]].head(5))

# 负深度处理（已确认）

以下规则已确认后再写入：`Depth < 0`（USGS 为 `depth`）置为空，并加布尔标记 `flag_depth_negative`。两边用同一规则。不删行，不导出清洗结果。


## STEP 27 · 本地负深度标记与置空

METHOD: 先按原值生成 `flag_depth_negative = Depth < 0`，再用 `mask` 把这些行的 `Depth` 置为空。不删行。


In [ ]:
n_local_before = len(df_local)
df_local["flag_depth_negative"] = df_local["Depth"] < 0
n_neg_local = int(df_local["flag_depth_negative"].sum())
df_local["Depth"] = df_local["Depth"].mask(df_local["flag_depth_negative"])
print("本地 flag_depth_negative 为 True 的行数:", n_neg_local)
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)

## STEP 28 · 检查本地负深度处理

METHOD: 确认处理后 `Depth < 0` 为 0，并列出被标记的行（`Date`、`Time`、处理后的 `Depth`、标记）。


In [ ]:
print("处理后 Depth < 0 的行数:", int((df_local["Depth"] < 0).sum()))
print("处理后 Depth 缺失数:", int(df_local["Depth"].isna().sum()))
print("处理后 Depth 最小值:", df_local["Depth"].min())
print()
print(df_local.loc[df_local["flag_depth_negative"], ["Date", "Time", "Depth", "flag_depth_negative"]])

## STEP 29 · USGS 负深度标记与置空

METHOD: 对 `df_usgs["depth"]` 使用同一规则：先加 `flag_depth_negative`，再把负值置空。即使当前没有负深度，也保留该列。不删行。


In [ ]:
n_usgs_before = len(df_usgs)
df_usgs["flag_depth_negative"] = df_usgs["depth"] < 0
n_neg_usgs = int(df_usgs["flag_depth_negative"].sum())
df_usgs["depth"] = df_usgs["depth"].mask(df_usgs["flag_depth_negative"])
print("USGS flag_depth_negative 为 True 的行数:", n_neg_usgs)
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)

## STEP 30 · 检查 USGS 负深度处理

METHOD: 确认处理后 `depth < 0` 为 0，并打印缺失数与最小值。


In [ ]:
print("处理后 depth < 0 的行数:", int((df_usgs["depth"] < 0).sum()))
print("处理后 depth 缺失数:", int(df_usgs["depth"].isna().sum()))
print("处理后 depth 最小值:", df_usgs["depth"].min())
print("flag_depth_negative 计数:")
print(df_usgs["flag_depth_negative"].value_counts())

# 类型标记（已确认）

以下规则已确认后再写入：非构造事件全部保留、不删行，只加布尔列 `is_earthquake`。本地按 `Type` 忽略大小写且恰好等于 `Earthquake`；USGS 按 `type` 忽略大小写且恰好等于 `earthquake`。不导出清洗结果。


## STEP 31 · 本地类型标记

METHOD: `is_earthquake = Type.str.lower() == "earthquake"`。不改 `Type`、不删行。


In [ ]:
n_local_before = len(df_local)
df_local["is_earthquake"] = df_local["Type"].astype(str).str.lower() == "earthquake"
print("本地 is_earthquake 为 True 的行数:", int(df_local["is_earthquake"].sum()))
print("本地 is_earthquake 为 False 的行数:", int((~df_local["is_earthquake"]).sum()))
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)

## STEP 32 · 检查本地类型标记

METHOD: 按 `Type` 与 `is_earthquake` 交叉计数，确认只有恰好等于 Earthquake 的行被标为 True。


In [ ]:
print(df_local.groupby(["Type", "is_earthquake"], dropna=False).size().rename("n"))

## STEP 33 · USGS 类型标记

METHOD: `is_earthquake = type.str.lower() == "earthquake"`。不改原始 `type`、不删行。


In [ ]:
n_usgs_before = len(df_usgs)
df_usgs["is_earthquake"] = df_usgs["type"].astype(str).str.lower() == "earthquake"
print("USGS is_earthquake 为 True 的行数:", int(df_usgs["is_earthquake"].sum()))
print("USGS is_earthquake 为 False 的行数:", int((~df_usgs["is_earthquake"]).sum()))
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)

## STEP 34 · 检查 USGS 类型标记

METHOD: 按 `type` 与 `is_earthquake` 交叉计数。


In [ ]:
print(df_usgs.groupby(["type", "is_earthquake"], dropna=False).size().rename("n"))

# 坐标检查（已确认）

以下规则已确认后再写入：纬度超出 [-90, 90] 或经度超出 [-180, 180] 的行加布尔标记 `flag_coord_invalid`。不改经纬度原值，不删行，不导出清洗结果。本地与 USGS 同一规则。


## STEP 35 · 本地坐标检查

METHOD: `flag_coord_invalid = (Latitude < -90) | (Latitude > 90) | (Longitude < -180) | (Longitude > 180)`。不改 `Latitude`/`Longitude`、不删行。


In [ ]:
n_local_before = len(df_local)
lat_before = df_local["Latitude"].copy()
lon_before = df_local["Longitude"].copy()
df_local["flag_coord_invalid"] = (
    (df_local["Latitude"] < -90)
    | (df_local["Latitude"] > 90)
    | (df_local["Longitude"] < -180)
    | (df_local["Longitude"] > 180)
)
print("本地 flag_coord_invalid 为 True 的行数:", int(df_local["flag_coord_invalid"].sum()))
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)
print("Latitude / Longitude 是否被改写:", (not df_local["Latitude"].equals(lat_before)) or (not df_local["Longitude"].equals(lon_before)))

## STEP 36 · 检查本地坐标标记

METHOD: 若有越界行，列出其经纬度与标记；否则只打印计数与当前范围。


In [ ]:
print("Latitude 范围:", df_local["Latitude"].min(), "–", df_local["Latitude"].max())
print("Longitude 范围:", df_local["Longitude"].min(), "–", df_local["Longitude"].max())
mask = df_local["flag_coord_invalid"]
if mask.any():
    print(df_local.loc[mask, ["Date", "Latitude", "Longitude", "flag_coord_invalid"]])
else:
    print("本地没有坐标越界行。")

## STEP 37 · USGS 坐标检查

METHOD: 对 `latitude`/`longitude` 使用同一规则，写入 `flag_coord_invalid`。不改原值、不删行。


In [ ]:
n_usgs_before = len(df_usgs)
lat_before = df_usgs["latitude"].copy()
lon_before = df_usgs["longitude"].copy()
df_usgs["flag_coord_invalid"] = (
    (df_usgs["latitude"] < -90)
    | (df_usgs["latitude"] > 90)
    | (df_usgs["longitude"] < -180)
    | (df_usgs["longitude"] > 180)
)
print("USGS flag_coord_invalid 为 True 的行数:", int(df_usgs["flag_coord_invalid"].sum()))
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)
print("latitude / longitude 是否被改写:", (not df_usgs["latitude"].equals(lat_before)) or (not df_usgs["longitude"].equals(lon_before)))

## STEP 38 · 检查 USGS 坐标标记

METHOD: 若有越界行则列出，否则只打印计数与当前范围。


In [ ]:
print("latitude 范围:", df_usgs["latitude"].min(), "–", df_usgs["latitude"].max())
print("longitude 范围:", df_usgs["longitude"].min(), "–", df_usgs["longitude"].max())
mask = df_usgs["flag_coord_invalid"]
if mask.any():
    print(df_usgs.loc[mask, ["time", "latitude", "longitude", "flag_coord_invalid"]])
else:
    print("USGS 没有坐标越界行。")

# 震级异常检查（已确认）

以下规则已确认后再写入：震级小于 0 或大于 9.5 的行加布尔标记 `flag_mag_outlier`。这不是完整度阈值。不改震级原值，不删行，不导出清洗结果。本地看 `Magnitude`，USGS 看 `mag`。


## STEP 39 · 本地震级异常检查

METHOD: `flag_mag_outlier = (Magnitude < 0) | (Magnitude > 9.5)`。不改 `Magnitude`、不删行。


In [ ]:
n_local_before = len(df_local)
mag_before = df_local["Magnitude"].copy()
df_local["flag_mag_outlier"] = (df_local["Magnitude"] < 0) | (df_local["Magnitude"] > 9.5)
print("本地 flag_mag_outlier 为 True 的行数:", int(df_local["flag_mag_outlier"].sum()))
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)
print("Magnitude 是否被改写:", not df_local["Magnitude"].equals(mag_before))

## STEP 40 · 检查本地震级标记

METHOD: 打印震级范围；若有异常行则列出，否则只打印计数。


In [ ]:
print("Magnitude 范围:", df_local["Magnitude"].min(), "–", df_local["Magnitude"].max())
mask = df_local["flag_mag_outlier"]
if mask.any():
    print(df_local.loc[mask, ["Date", "Magnitude", "flag_mag_outlier"]])
else:
    print("本地没有震级异常行。")

## STEP 41 · USGS 震级异常检查

METHOD: 对 `mag` 使用同一规则，写入 `flag_mag_outlier`。不改原值、不删行。


In [ ]:
n_usgs_before = len(df_usgs)
mag_before = df_usgs["mag"].copy()
df_usgs["flag_mag_outlier"] = (df_usgs["mag"] < 0) | (df_usgs["mag"] > 9.5)
print("USGS flag_mag_outlier 为 True 的行数:", int(df_usgs["flag_mag_outlier"].sum()))
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)
print("mag 是否被改写:", not df_usgs["mag"].equals(mag_before))

## STEP 42 · 检查 USGS 震级标记

METHOD: 打印震级范围；若有异常行则列出，否则只打印计数。


In [ ]:
print("mag 范围:", df_usgs["mag"].min(), "–", df_usgs["mag"].max())
mask = df_usgs["flag_mag_outlier"]
if mask.any():
    print(df_usgs.loc[mask, ["time", "mag", "flag_mag_outlier"]])
else:
    print("USGS 没有震级异常行。")

# 深度上限检查（已确认）

以下规则已确认后再写入：深度大于 700 公里的行加布尔标记 `flag_depth_over_700`。等于 700 不标记。不改深度原值，不删行，不导出清洗结果。本地看处理后的 `Depth`，USGS 看 `depth`。


## STEP 43 · 本地深度上限检查

METHOD: `flag_depth_over_700 = Depth > 700`。不改 `Depth`、不删行。


In [ ]:
n_local_before = len(df_local)
depth_before = df_local["Depth"].copy()
df_local["flag_depth_over_700"] = df_local["Depth"] > 700
print("本地 flag_depth_over_700 为 True 的行数:", int(df_local["flag_depth_over_700"].sum()))
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)
print("Depth 是否被改写:", not df_local["Depth"].equals(depth_before))

## STEP 44 · 检查本地深度上限标记

METHOD: 打印深度范围；若有超限行则列出，否则只打印计数。


In [ ]:
print("Depth 范围:", df_local["Depth"].min(), "–", df_local["Depth"].max())
print("Depth == 700 的行数:", int((df_local["Depth"] == 700).sum()))
mask = df_local["flag_depth_over_700"]
if mask.any():
    print(df_local.loc[mask, ["Date", "Depth", "flag_depth_over_700"]])
else:
    print("本地没有深度大于 700 的行。")

## STEP 45 · USGS 深度上限检查

METHOD: 对 `depth` 使用同一规则，写入 `flag_depth_over_700`。不改原值、不删行。


In [ ]:
n_usgs_before = len(df_usgs)
depth_before = df_usgs["depth"].copy()
df_usgs["flag_depth_over_700"] = df_usgs["depth"] > 700
print("USGS flag_depth_over_700 为 True 的行数:", int(df_usgs["flag_depth_over_700"].sum()))
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)
print("depth 是否被改写:", not df_usgs["depth"].equals(depth_before))

## STEP 46 · 检查 USGS 深度上限标记

METHOD: 打印深度范围；若有超限行则列出，否则只打印计数。


In [ ]:
print("depth 范围:", df_usgs["depth"].min(), "–", df_usgs["depth"].max())
mask = df_usgs["flag_depth_over_700"]
if mask.any():
    print(df_usgs.loc[mask, ["time", "depth", "flag_depth_over_700"]])
else:
    print("USGS 没有深度大于 700 的行。")

# 疑似重复检查（已确认）

以下规则已确认后再写入：同一数据源内比较；时间取整到最近一分钟后相等，球面距离小于 50 公里，且震级差的绝对值小于 0.3，则视为一组，组内全部标为 `flag_possible_duplicate`。只加标记，不删行，不改已有的 `Date_parsed`。本地大多数 `Date_parsed` 只有日期（0 点），比较时用 `Date` 对应的日期与可解析的 `Time` 拼出完整时间；USGS 的 `Date_parsed` 已含时分秒，直接取整。`Date_parsed` 缺失、坐标越界或震级缺失的行不参与配对。不导出清洗结果。


## STEP 47 · 定义重复检查函数

METHOD: `haversine_km` 计算球面距离。本地在 `Date_parsed` 为午夜、且 `Time` 可按 `HH:MM:SS` 解析为时长时，把 `Time` 加到日期上得到比较用时间，不写回 `Date_parsed`。USGS 无 `Time` 列，直接使用 `Date_parsed`。再按分钟分组，用并查集把距离 < 50 km 且 |Δ震级| < 0.3 的记录连成一组。


In [ ]:
import numpy as np
from collections import defaultdict


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def build_dup_timestamp(df):
    ts = pd.to_datetime(df["Date_parsed"], utc=True, errors="coerce")
    if "Time" not in df.columns:
        return ts
    td = pd.to_timedelta(df["Time"].astype(str), errors="coerce")
    midnight = (
        (ts.dt.hour == 0) & (ts.dt.minute == 0)
        & (ts.dt.second == 0) & (ts.dt.microsecond == 0)
    )
    use = midnight & td.notna() & ts.notna()
    ts = ts.copy()
    ts.loc[use] = ts.loc[use] + td.loc[use]
    return ts


def add_duplicate_flag(df, lat_col, lon_col, mag_col):
    out = df.copy()
    out["flag_possible_duplicate"] = False
    ts = build_dup_timestamp(out)
    minute = ts.dt.round("min")
    valid = ts.notna() & ~out["flag_coord_invalid"]
    tmp = out.loc[valid, [lat_col, lon_col, mag_col]].copy()
    tmp["_minute"] = minute.loc[valid]

    parent = {i: i for i in tmp.index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for _, grp in tmp.groupby("_minute"):
        if len(grp) < 2:
            continue
        idx = grp.index.to_list()
        lats = grp[lat_col].to_numpy(dtype=float)
        lons = grp[lon_col].to_numpy(dtype=float)
        mags = grp[mag_col].to_numpy(dtype=float)
        n = len(grp)
        for i in range(n):
            for j in range(i + 1, n):
                if np.isnan(mags[i]) or np.isnan(mags[j]):
                    continue
                if abs(mags[i] - mags[j]) >= 0.3:
                    continue
                dist = haversine_km(lats[i], lons[i], lats[j], lons[j])
                if dist < 50:
                    union(idx[i], idx[j])

    comps = defaultdict(list)
    for i in tmp.index:
        comps[find(i)].append(i)
    n_groups = 0
    n_rows = 0
    for members in comps.values():
        if len(members) >= 2:
            out.loc[members, "flag_possible_duplicate"] = True
            n_groups += 1
            n_rows += len(members)
    return out, n_groups, n_rows

## STEP 48 · 本地疑似重复标记

METHOD: 对 `df_local` 调用上述函数，经纬度与震级使用 `Latitude`、`Longitude`、`Magnitude`。比较用时间不写回 `Date_parsed`。不删行。


In [ ]:
n_local_before = len(df_local)
date_parsed_before = df_local["Date_parsed"].copy()
df_local, n_dup_g_local, n_dup_r_local = add_duplicate_flag(
    df_local, "Latitude", "Longitude", "Magnitude"
)
print("本地疑似重复组数:", n_dup_g_local)
print("本地被标记行数:", n_dup_r_local)
print("行数是否变化:", n_local_before, "→", len(df_local), "变化量:", len(df_local) - n_local_before)
print("Date_parsed 是否被改写:", not df_local["Date_parsed"].equals(date_parsed_before))

## STEP 49 · 检查本地疑似重复

METHOD: 打印标记计数，并列出若干被标记行的 `Date`、`Time`、经纬度与震级，便于人工抽查。


In [ ]:
print("本地 flag_possible_duplicate 计数:")
print(df_local["flag_possible_duplicate"].value_counts())
cols = ["Date", "Time", "Latitude", "Longitude", "Magnitude", "flag_possible_duplicate"]
if n_dup_r_local:
    print()
    print(df_local.loc[df_local["flag_possible_duplicate"], cols].head(8))
else:
    print("本地没有疑似重复行。")

## STEP 50 · USGS 疑似重复标记

METHOD: 对 `df_usgs` 使用同一规则，经纬度与震级使用 `latitude`、`longitude`、`mag`。USGS 无 `Time` 列，比较时间直接取 `Date_parsed`。不改 `Date_parsed`、不删行。


In [ ]:
n_usgs_before = len(df_usgs)
date_parsed_before = df_usgs["Date_parsed"].copy()
df_usgs, n_dup_g_usgs, n_dup_r_usgs = add_duplicate_flag(
    df_usgs, "latitude", "longitude", "mag"
)
print("USGS 疑似重复组数:", n_dup_g_usgs)
print("USGS 被标记行数:", n_dup_r_usgs)
print("行数是否变化:", n_usgs_before, "→", len(df_usgs), "变化量:", len(df_usgs) - n_usgs_before)
print("Date_parsed 是否被改写:", not df_usgs["Date_parsed"].equals(date_parsed_before))

## STEP 51 · 检查 USGS 疑似重复

METHOD: 打印标记计数，并列出被标记行的 `time`、经纬度与震级。


In [ ]:
print("USGS flag_possible_duplicate 计数:")
print(df_usgs["flag_possible_duplicate"].value_counts())
cols = ["time", "latitude", "longitude", "mag", "flag_possible_duplicate"]
if n_dup_r_usgs:
    print()
    print(df_usgs.loc[df_usgs["flag_possible_duplicate"], cols])
else:
    print("USGS 没有疑似重复行。")

# 逆地理（已确认，全球精细化）

以下规则已确认后再写入：用离线 `reverse_geocoder` 为所有记录映射国家，并统一取出一级行政区；中国另留省份列。再用 Natural Earth 10m 一级行政区多边形做点面核对：仅当点落在**同一国家**的另一行政区内、且距原标注行政区超过 100 公里时，写入校正表并改正 `Admin1`（中国同时改正 `Province`）。靠近边界（≤100 km）或海上、未匹配到多边形名称、以及会改到另一国家的个例不改。原始值保留在 `*_raw` 列。7 个未匹配领地代码（GP、SJ、GF、RE、CC、CX、MQ）加 `flag_territory_unmatched`。不删行。本步会写出校正表 `07_geocode_corrections.csv`，但不导出完整清洗数据。


## STEP 52 · 逆地理依赖与边界数据路径

METHOD: 导入 `reverse_geocoder`、`geopandas`、`pyproj`。行政区多边形优先使用本项目已有的 `mapdata/ne_10m_admin_1_states_provinces`（在 `07_fine_clean` 的上一级，或箱子上的副本）。


In [ ]:
import re
import unicodedata
import reverse_geocoder as rg
import geopandas as gpd
from shapely.ops import nearest_points
from pyproj import Geod

MAP_CANDIDATES = [
    ROOT.parent / "mapdata" / "ne_10m_admin_1_states_provinces" / "ne_10m_admin_1_states_provinces.shp",
    Path(r"C:\Users\WHB\Desktop\DPW I\mapdata\ne_10m_admin_1_states_provinces\ne_10m_admin_1_states_provinces.shp"),
    Path("/workspace/dpw1_report/mapdata/ne_10m_admin_1_states_provinces/ne_10m_admin_1_states_provinces.shp"),
    ROOT / "mapdata" / "ne_10m_admin_1_states_provinces" / "ne_10m_admin_1_states_provinces.shp",
]
SHP_ADMIN1 = next((p for p in MAP_CANDIDATES if p.exists()), None)
if SHP_ADMIN1 is None:
    raise FileNotFoundError(
        "未找到 ne_10m_admin_1_states_provinces.shp，请确认项目 mapdata 文件夹仍在。"
    )
CORR_OUT = ROOT / "07_geocode_corrections.csv"
TERRITORY_UNMATCHED = {"GP", "SJ", "GF", "RE", "CC", "CX", "MQ"}
print("admin1 shp:", SHP_ADMIN1)
print("correction out:", CORR_OUT)

## STEP 53 · 离线逆地理（国家与一级行政区）

METHOD: `reverse_geocoder.search(..., mode=1)` 批量离线查询。`Country_raw` ← `cc`；所有国家的 `Admin1_raw` ← `admin1`；仅当 `cc == "CN"` 时把同一 `admin1` 写入 `Province_raw`。坐标缺失或 `flag_coord_invalid` 的行留空。不删行。


In [ ]:
def add_geocode(df, lat_col, lon_col):
    out = df.copy()
    valid = out[lat_col].notna() & out[lon_col].notna() & ~out["flag_coord_invalid"]
    country = pd.Series(np.nan, index=out.index, dtype="object")
    admin1 = pd.Series(np.nan, index=out.index, dtype="object")
    if valid.any():
        coords = list(zip(
            out.loc[valid, lat_col].astype(float),
            out.loc[valid, lon_col].astype(float),
        ))
        results = rg.search(coords, mode=1)
        cc_list, adm_list = [], []
        for rec in results:
            cc = rec.get("cc") or np.nan
            adm = rec.get("admin1")
            if adm is None or str(adm).strip() == "":
                adm = np.nan
            cc_list.append(cc)
            adm_list.append(adm)
        country.loc[valid] = cc_list
        admin1.loc[valid] = adm_list
    out["Country_raw"] = country
    out["Admin1_raw"] = admin1
    out["Province_raw"] = np.where(out["Country_raw"] == "CN", out["Admin1_raw"], np.nan)
    return out

print("geocoding local ...")
n_local_before = len(df_local)
df_local = add_geocode(df_local, "Latitude", "Longitude")
print("geocoding usgs ...")
n_usgs_before = len(df_usgs)
df_usgs = add_geocode(df_usgs, "latitude", "longitude")
print("行数是否变化: local", n_local_before, "→", len(df_local), "; usgs", n_usgs_before, "→", len(df_usgs))
print("local Country 非空:", int(df_local["Country_raw"].notna().sum()),
      "Admin1 非空:", int(df_local["Admin1_raw"].notna().sum()),
      "CN:", int((df_local["Country_raw"] == "CN").sum()))
print("usgs  Country 非空:", int(df_usgs["Country_raw"].notna().sum()),
      "Admin1 非空:", int(df_usgs["Admin1_raw"].notna().sum()),
      "CN:", int((df_usgs["Country_raw"] == "CN").sum()))

## STEP 54 · 名称规范化，并生成中国省份列

METHOD: `Country` 先等于 `Country_raw`。`Admin1` 对全球一级行政区做轻度规范化（去掉常见后缀；中国用短英文名，如 `Qinghai Sheng` → `Qinghai`，`Tibet Autonomous Region` → `Tibet`）。`Province` 仅在 `Country_raw == "CN"` 时等于规范化后的 `Admin1`，否则为空。


In [ ]:
PROVINCE_NAME_MAP = {
    "Xinjiang Uygur Zizhiqu": "Xinjiang",
    "Tibet Autonomous Region": "Tibet",
    "Xizang": "Tibet",
    "Qinghai Sheng": "Qinghai",
    "Heilongjiang Sheng": "Heilongjiang",
    "Gansu Sheng": "Gansu",
    "Jiangsu Sheng": "Jiangsu",
    "Inner Mongolia": "Inner Mongolia",
    "Inner Mongol": "Inner Mongolia",
    "Tianjin Shi": "Tianjin",
    "Shandong Sheng": "Shandong",
    "Shanxi Sheng": "Shanxi",
    "Jilin Sheng": "Jilin",
    "Zhejiang Sheng": "Zhejiang",
    "Guangxi Zhuangzu Zizhiqu": "Guangxi",
    "Ningxia Huizu Zizhiqu": "Ningxia",
    "Beijing Shi": "Beijing",
    "Shanghai Shi": "Shanghai",
    "Chongqing Shi": "Chongqing",
}

def map_admin1_name(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return np.nan
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return np.nan
    if text in PROVINCE_NAME_MAP:
        return PROVINCE_NAME_MAP[text]
    out = text
    for token in [
        " Uygur Zizhiqu", " Huizu Zizhiqu", " Zhuangzu Zizhiqu",
        " Autonomous Region", " Zizhiqu", " Sheng", " Shi",
        " Province", " Municipality",
    ]:
        out = out.replace(token, "")
    out = " ".join(out.split())
    return out or text

df_local["Country"] = df_local["Country_raw"]
df_usgs["Country"] = df_usgs["Country_raw"]
df_local["Admin1"] = df_local["Admin1_raw"].map(map_admin1_name)
df_usgs["Admin1"] = df_usgs["Admin1_raw"].map(map_admin1_name)
df_local["Province"] = np.where(df_local["Country_raw"] == "CN", df_local["Admin1"], np.nan)
df_usgs["Province"] = np.where(df_usgs["Country_raw"] == "CN", df_usgs["Admin1"], np.nan)
print("local CN Province（校正前）:")
print(df_local.loc[df_local["Country_raw"] == "CN", "Province"].value_counts(dropna=False).to_string())

## STEP 55 · 载入全球一级行政区多边形并建立名称索引

METHOD: 读取 Natural Earth 10m admin1。用 `name` / `name_en` / `gn_name` / `woe_name` / `name_alt` 以及轻度规范化、少量音译别名（如 Kamtsjatka → Kamchatka）建立 `(国家代码, 名称)` 到多边形的索引，供下一步点面核对。


In [ ]:
SUFFIXES = [
    "uygur zizhiqu", "huizu zizhiqu", "zhuangzu zizhiqu", "autonomous region",
    "zizhiqu", "sheng", "shi", "province", "municipality", "prefecture",
    "oblast", "krai", "department", "departamento", "region", "région",
    "state", "county", "district", "territory", "governorate", "provinsi", "daerah",
]
ADMIN1_ALIASES = {
    ("RU", "kamtsjatka"): "kamchatka",
    ("RU", "primorskiy"): "primorye",
    ("RU", "sakha"): "sakha yakutia",
    ("TF", "kerguelen"): "kerguelen islands",
    ("IR", "azarbayjan e gharbi"): "west azarbaijan",
}

def base_norm(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    t = str(s).strip()
    if t == "" or t.lower() == "nan":
        return None
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = t.lower().replace("'", "").replace("’", "")
    t = re.sub(r"[^a-z0-9]+", " ", t).strip()
    return t or None

def strip_suffixes(t):
    if not t:
        return t
    changed = True
    while changed:
        changed = False
        for suf in SUFFIXES:
            if t.endswith(" " + suf):
                t = t[: -(len(suf) + 1)].strip()
                changed = True
    return t or None

def name_variants(s, iso=None):
    b = base_norm(s)
    if not b:
        return set()
    out = {b}
    ss = strip_suffixes(b)
    if ss:
        out.add(ss)
    extra = {"xizang": "tibet", "inner mongol": "inner mongolia"}
    for v in list(out):
        if iso and (iso, v) in ADMIN1_ALIASES:
            out.add(ADMIN1_ALIASES[(iso, v)])
        if v in extra:
            out.add(extra[v])
    return {x for x in out if x}

admin1_gdf = gpd.read_file(SHP_ADMIN1)
print("admin1 polygons:", len(admin1_gdf), "crs:", admin1_gdf.crs)

ADMIN1_LOOKUP = {}
for i, row in admin1_gdf.iterrows():
    iso = row.get("iso_a2")
    if not iso or iso == "-99":
        continue
    names = []
    for col in ["name", "name_en", "gn_name", "woe_name", "name_alt"]:
        val = row.get(col)
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        names.extend(str(val).split("|"))
    for n in names:
        for v in name_variants(n, iso):
            ADMIN1_LOOKUP.setdefault((iso, v), set()).add(i)

def resolve_admin_idx(cc, adm):
    if pd.isna(cc) or pd.isna(adm):
        return set()
    hits = set()
    for v in name_variants(adm, cc):
        hits |= ADMIN1_LOOKUP.get((cc, v), set())
    return hits

print("name index keys:", len(ADMIN1_LOOKUP))

## STEP 56 · 点面核对并计算偏离距离

METHOD: 将事件点与行政区多边形做 `intersects`。若点落在原标注行政区内，视为正确。否则计算点到原标注多边形边界的测地线距离。此步只生成核对结果，先不改正。


In [ ]:
GEOD = Geod(ellps="WGS84")

def distance_km_to_polys(point, idxs):
    dmin = np.inf
    for idx in idxs:
        poly = admin1_gdf.loc[idx].geometry
        if poly is None or poly.is_empty:
            continue
        if poly.covers(point):
            return 0.0
        p, q = nearest_points(point, poly)
        _, _, d = GEOD.inv(p.x, p.y, q.x, q.y)
        dmin = min(dmin, abs(d) / 1000.0)
    return dmin

def audit_admin1(df, lat_col, lon_col):
    pts = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
        index=pd.RangeIndex(len(df)),
    )
    pts["_i"] = np.arange(len(df))
    tmp = admin1_gdf[["iso_a2", "name", "name_en", "geometry"]].copy()
    tmp["_idx"] = tmp.index
    joined = pts.sjoin(tmp, predicate="intersects", how="left")
    contain = joined.groupby("_i")["_idx"].apply(lambda s: set(s.dropna().astype(int)))
    contain_names = joined.groupby("_i")[["iso_a2", "name_en", "name"]].apply(
        lambda x: list(zip(x["iso_a2"].tolist(), x["name_en"].fillna(x["name"]).tolist()))
    )
    records = []
    stats = {"ok": 0, "name_unmatched": 0, "border": 0, "offshore": 0, "cross_country": 0, "correctable": 0}
    for pos in range(len(df)):
        row = df.iloc[pos]
        labeled = resolve_admin_idx(row["Country_raw"], row["Admin1_raw"])
        inside = contain.get(pos, set())
        names = contain_names.get(pos, [])
        if labeled and (labeled & inside):
            stats["ok"] += 1
            continue
        if not labeled:
            stats["name_unmatched"] += 1
            continue
        dist = distance_km_to_polys(pts.geometry.iloc[pos], labeled)
        if dist <= 100:
            stats["border"] += 1
            continue
        if not inside:
            stats["offshore"] += 1
            continue
        same = [(iso, nm) for iso, nm in names if iso == row["Country_raw"] and nm and str(nm) != "nan"]
        if not same:
            stats["cross_country"] += 1
            continue
        iso, nm = same[0]
        stats["correctable"] += 1
        records.append({
            "pos": pos,
            "Country_raw": row["Country_raw"],
            "Admin1_raw": row["Admin1_raw"],
            "Country_corrected": iso,
            "Admin1_corrected": map_admin1_name(nm),
            "dist_km": round(float(dist), 1),
        })
    return records, stats

print("auditing local ...")
recs_local, stats_local = audit_admin1(df_local, "Latitude", "Longitude")
print("local", stats_local)
print("auditing usgs ...")
recs_usgs, stats_usgs = audit_admin1(df_usgs, "latitude", "longitude")
print("usgs ", stats_usgs)

## STEP 57 · 生成校正表并保存

METHOD: 把上一步判定为可校正的记录写成 `07_geocode_corrections.csv`（同国、落入另一行政区、偏离 > 100 km）。原始 `*_raw` 一并保留。不导出完整清洗数据。


In [ ]:
def records_to_table(df, recs, dataset, lat_col, lon_col, mag_col, date_col, time_col=None):
    rows = []
    for rec in recs:
        r = df.iloc[rec["pos"]]
        reason = f"坐标位于{rec['Admin1_corrected']}，但逆地理误标为{rec['Admin1_raw']}；偏离约 {rec['dist_km']} km"
        if (
            rec["Country_raw"] == "CN"
            and rec["Admin1_corrected"] == "Qinghai"
            and abs(float(r[lat_col]) - 35.946) < 1e-3
            and abs(float(r[lon_col]) - 90.541) < 1e-3
        ):
            reason += "；2001年昆仑山口西 M7.8"
        rows.append({
            "dataset": dataset,
            "Date": r[date_col],
            "Time": (r[time_col] if time_col else np.nan),
            "Latitude": r[lat_col],
            "Longitude": r[lon_col],
            "Magnitude": r[mag_col],
            "Country_raw": rec["Country_raw"],
            "Admin1_raw": rec["Admin1_raw"],
            "Province_raw": r.get("Province_raw", np.nan) if rec["Country_raw"] == "CN" else np.nan,
            "Country_corrected": rec["Country_corrected"],
            "Admin1_corrected": rec["Admin1_corrected"],
            "Province_corrected": (rec["Admin1_corrected"] if rec["Country_raw"] == "CN" else np.nan),
            "dist_km": rec["dist_km"],
            "reason": reason,
        })
    return pd.DataFrame(rows)

corr_local = records_to_table(df_local, recs_local, "local", "Latitude", "Longitude", "Magnitude", "Date", "Time")
corr_usgs = records_to_table(df_usgs, recs_usgs, "usgs", "latitude", "longitude", "mag", "time")
corr = pd.concat([corr_local, corr_usgs], ignore_index=True)
corr.to_csv(CORR_OUT, index=False)
print("wrote:", CORR_OUT, "rows:", len(corr))
print(corr["dataset"].value_counts().to_string())
print("按原国家计数:")
print(corr["Country_raw"].value_counts().to_string())
print("中国校正:")
print(corr.loc[corr["Country_raw"] == "CN",
               ["dataset", "Date", "Admin1_raw", "Admin1_corrected", "dist_km"]].to_string())

## STEP 58 · 应用校正

METHOD: 按 `dataset` + 经纬度（容差 1e-4°）+ 震级（容差 1e-3）+ 日期匹配。命中则改 `Admin1`（中国同时改 `Province`），需要时改 `Country`，并设 `flag_geocode_corrected=True`。`*_raw` 列保持逆地理原值。不删行。


In [ ]:
def apply_corrections(df, corr_df, dataset_name, lat_col, lon_col, mag_col, date_col):
    out = df.copy()
    out["flag_geocode_corrected"] = False
    sub = corr_df.loc[corr_df["dataset"] == dataset_name]
    n_applied = 0
    n_unmatched = 0
    lat = pd.to_numeric(out[lat_col], errors="coerce")
    lon = pd.to_numeric(out[lon_col], errors="coerce")
    mag = pd.to_numeric(out[mag_col], errors="coerce")
    for _, row in sub.iterrows():
        mask = (
            np.isclose(lat, float(row["Latitude"]), atol=1e-4, equal_nan=False)
            & np.isclose(lon, float(row["Longitude"]), atol=1e-4, equal_nan=False)
            & np.isclose(mag, float(row["Magnitude"]), atol=1e-3, equal_nan=False)
            & (out[date_col].astype(str) == str(row["Date"]))
        )
        n = int(mask.sum())
        if n == 0:
            n_unmatched += 1
            print("  unmatched correction:", dict(row[["dataset", "Date", "Latitude", "Longitude", "Magnitude"]]))
            continue
        out.loc[mask, "Admin1"] = row["Admin1_corrected"]
        if pd.notna(row["Country_corrected"]):
            out.loc[mask, "Country"] = row["Country_corrected"]
        if pd.notna(row["Province_corrected"]):
            out.loc[mask, "Province"] = row["Province_corrected"]
        out.loc[mask, "flag_geocode_corrected"] = True
        n_applied += n
    print(f"{dataset_name}: applied {n_applied} / {len(sub)} (unmatched {n_unmatched})")
    return out

n_local_before = len(df_local)
n_usgs_before = len(df_usgs)
df_local = apply_corrections(df_local, corr, "local", "Latitude", "Longitude", "Magnitude", "Date")
df_usgs = apply_corrections(df_usgs, corr, "usgs", "latitude", "longitude", "mag", "time")
print("行数是否变化: local", n_local_before, "→", len(df_local), "; usgs", n_usgs_before, "→", len(df_usgs))

## STEP 59 · 领地未匹配标记

METHOD: 若 `Country` 属于 `GP, SJ, GF, RE, CC, CX, MQ`，则 `flag_territory_unmatched=True`。不删行。


In [ ]:
df_local["flag_territory_unmatched"] = df_local["Country"].isin(TERRITORY_UNMATCHED)
df_usgs["flag_territory_unmatched"] = df_usgs["Country"].isin(TERRITORY_UNMATCHED)
print("local territory unmatched:", int(df_local["flag_territory_unmatched"].sum()))
print("usgs  territory unmatched:", int(df_usgs["flag_territory_unmatched"].sum()))

## STEP 60 · 检查逆地理与校正结果

METHOD: 核对 2001 年昆仑山口西 M7.8 是否已由黑龙江改为青海；并汇总校正计数、中国省份分布。


In [ ]:
kunlun = df_local[
    np.isclose(pd.to_numeric(df_local["Latitude"], errors="coerce"), 35.946, atol=1e-3)
    & np.isclose(pd.to_numeric(df_local["Longitude"], errors="coerce"), 90.541, atol=1e-3)
]
print("Kunlun check:")
print(kunlun[["Date", "Time", "Latitude", "Longitude", "Magnitude",
              "Country_raw", "Admin1_raw", "Province_raw",
              "Country", "Admin1", "Province", "flag_geocode_corrected"]].to_string())
print()
print("local 校正行数:", int(df_local["flag_geocode_corrected"].sum()))
print("usgs  校正行数:", int(df_usgs["flag_geocode_corrected"].sum()))
print("local CN Province（校正后）:")
print(df_local.loc[df_local["Country"] == "CN", "Province"].value_counts(dropna=False).to_string())
print()
print("local Admin1 校正样本:")
cols_l = ["Date", "Country_raw", "Admin1_raw", "Country", "Admin1", "flag_geocode_corrected"]
print(df_local.loc[df_local["flag_geocode_corrected"], cols_l].head(8).to_string())

# 导出清洗结果（已确认）

以下规则已确认后再写入：在本文件夹下新建 `cleaned/`，分别导出本地与 USGS 的精细清洗结果。不覆盖原项目中的 `ProjectData`、`acquired` 以及旧的清洗文件。


## STEP 61 · 导出精细清洗 CSV

METHOD: 在 `07_fine_clean/cleaned/` 下用 `to_csv(index=False)` 写出 `07_earthquake_fine_cleaned.csv` 与 `07_usgs_quakes_fine_cleaned.csv`。不删行，不改已有列。


In [ ]:
OUT = ROOT / "cleaned"
OUT.mkdir(exist_ok=True)
LOCAL_OUT = OUT / "07_earthquake_fine_cleaned.csv"
USGS_OUT = OUT / "07_usgs_quakes_fine_cleaned.csv"

n_local_before = len(df_local)
n_usgs_before = len(df_usgs)
df_local.to_csv(LOCAL_OUT, index=False)
df_usgs.to_csv(USGS_OUT, index=False)
print("wrote:", LOCAL_OUT, "rows:", n_local_before)
print("wrote:", USGS_OUT, "rows:", n_usgs_before)

## STEP 62 · 检查导出文件

METHOD: 重新读回刚写出的 CSV，核对行数与关键列是否还在，并确认原始数据文件没有被覆盖。


In [ ]:
local_chk = pd.read_csv(LOCAL_OUT)
usgs_chk = pd.read_csv(USGS_OUT)
print("回读 local:", local_chk.shape, "期望", (n_local_before, df_local.shape[1]))
print("回读 usgs :", usgs_chk.shape, "期望", (n_usgs_before, df_usgs.shape[1]))
print("行数一致:", len(local_chk) == n_local_before and len(usgs_chk) == n_usgs_before)

need_local = ["Date", "Date_parsed", "is_earthquake", "flag_depth_negative",
              "flag_possible_duplicate", "Country", "Admin1", "Province",
              "flag_geocode_corrected", "flag_territory_unmatched"]
need_usgs = ["time", "Date_parsed", "is_earthquake", "flag_depth_negative",
             "flag_possible_duplicate", "Country", "Admin1", "Province",
             "flag_geocode_corrected", "flag_territory_unmatched"]
print("local 关键列齐全:", all(c in local_chk.columns for c in need_local))
print("usgs  关键列齐全:", all(c in usgs_chk.columns for c in need_usgs))
print("原始本地文件仍在且未被本步作为输出:", LOCAL_IN.exists(), LOCAL_IN)
print("原始 USGS 文件仍在且未被本步作为输出:", USGS_IN.exists(), USGS_IN)